<img src="imagens/banner.png" alt="Módulo 03: A biblioteca requests" width="1100">

# Módulo 03: A biblioteca requests

No Módulo 02, você viu como cliente e servidor conversam. Neste módulo, o Python assume o papel de cliente com a biblioteca **requests**, a mais usada para fazer pedidos HTTP.

<img src="imagens/mapa-do-curso.png" alt="Mapa dos 10 módulos do curso, com o Módulo 03 destacado" width="1100">

## Neste módulo você vai

- Fazer pedidos com `requests.get()` e explorar a resposta, atributo por atributo;
- Ler o conteúdo como texto, bytes ou JSON, e corrigir problemas de **codificação**;
- Conferir o **código de status** e tratar erros: `404`, falhas de conexão e tempo esgotado;
- Enviar **parâmetros** e **cabeçalhos**, e entender o papel do `User-Agent`;
- Usar **cookies** e **sessões** para fazer login em um site;
- Baixar o **catálogo inteiro** de um site, página por página, de forma responsável.

**Tempo estimado:** de 1h30 a 2h.

## Como este módulo funciona

O módulo avança em passos pequenos: uma explicação curta e, logo abaixo, uma célula curta que faz exatamente o que foi explicado. Execute cada uma antes de ler a próxima explicação.

Os exercícios, marcados como **Sua vez**, também são divididos em passos. Cada passo tem a sua célula e a sua resposta, que fica oculta logo abaixo: tente resolver antes de consultá-la.

**Antes de começar:** abra o ambiente do curso (rotina do Módulo 01) e abra este notebook, `03-requests.ipynb`.

▶️ Comece importando a biblioteca. A célula também mostra a versão instalada.

In [ ]:
import requests

requests.__version__

---

# Parte 1: Pedidos e respostas

## 1.1 O primeiro pedido

A função `requests.get()` faz um pedido `GET` e devolve um objeto **resposta** (`Response`), que reúne tudo o que o servidor enviou:

<img src="imagens/requests-get.png" alt="O código de requests.get com URL, params, headers e timeout, e o objeto resposta com seus atributos agrupados" width="1120">

O argumento `timeout` define quantos segundos esperar por uma resposta. Ele deve estar em todos os pedidos: sem ele, um servidor que não responde trava o programa.

▶️ Faça o pedido. O resultado é guardado na variável `resposta`, e a célula não mostra nada.

In [ ]:
resposta = requests.get("https://books.toscrape.com/", timeout=20)

A partir de agora, a variável `resposta` **contém tudo o que voltou do servidor**: o código de status, os cabeçalhos, o conteúdo da página e até uma cópia do pedido que foi feito. Cada informação fica em um **atributo**, acessado com um ponto depois do nome da variável, como `resposta.status_code`.

▶️ Primeiro, veja o que é a variável. Digitar só o nome mostra um resumo dela:

In [ ]:
resposta

O resumo `<Response [200]>` diz que é uma resposta, com o código 200.

▶️ O código fica no atributo `status_code`, e a descrição, no `reason`:

In [ ]:
print(resposta.status_code, resposta.reason)

▶️ O atributo `ok` resume o resultado: é `True` para qualquer código menor que 400.

In [ ]:
resposta.ok

▶️ Os cabeçalhos recebidos ficam em `headers`, que funciona como um dicionário. Consulte o tipo do conteúdo:

In [ ]:
resposta.headers["Content-Type"]

▶️ O conteúdo da página fica em `text`: é o HTML, em forma de texto. Ele é longo, então mostre só os primeiros 300 caracteres:

In [ ]:
print(resposta.text[:300])

As primeiras linhas, que começam com `<!--[if`, são instruções para navegadores antigos, como você viu no Módulo 02.

Por enquanto, esse HTML é só uma longa sequência de caracteres. No Módulo 04, ele será entregue ao **parsel**, uma biblioteca que entende a **estrutura** do HTML (a árvore do DOM, vista no Módulo 02) e permite ir direto a cada dado com precisão.

▶️ Uma prévia: a célula abaixo entrega o `text` ao parsel e pede o título da página. Não se preocupe com os detalhes agora.

In [ ]:
from parsel import Selector

pagina = Selector(text=resposta.text)
pagina.css("title::text").get().strip()  # strip() tira os espaços das pontas

▶️ A resposta também guarda o **pedido** que a originou, no atributo `request`. Veja o método e o endereço:

In [ ]:
print(resposta.request.method, resposta.request.url)

▶️ Por fim, `elapsed` mostra quanto tempo o servidor levou para responder:

In [ ]:
resposta.elapsed.total_seconds()

### Sua vez: o seu primeiro pedido

**Passo 1 de 3.** Faça um pedido para `https://quotes.toscrape.com/` e guarde o resultado em uma variável chamada `citacoes`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
citacoes = requests.get("https://quotes.toscrape.com/", timeout=20)
```
</details>

**Passo 2 de 3.** Mostre o código de status de `citacoes`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
citacoes.status_code
```
</details>

**Passo 3 de 3.** Mostre os primeiros 150 caracteres do texto de `citacoes`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
print(citacoes.text[:150])
```
</details>

## 1.2 Três formas de ler o conteúdo

O corpo da resposta pode ser lido de três formas, conforme o tipo do conteúdo:

| Atributo | Devolve | Use para |
|---|---|---|
| `resposta.text` | texto (`str`) | Páginas HTML e outros textos |
| `resposta.content` | bytes (`bytes`) | Imagens, PDFs e outros arquivos |
| `resposta.json()` | dicionário ou lista | Respostas em JSON, o formato de dados das APIs |

### Bytes: baixando uma imagem

▶️ Os arquivos baixados neste módulo ficam em uma pasta chamada `dados`. Crie a pasta (se ela já existir, nada acontece):

In [ ]:
from pathlib import Path

pasta = Path("dados")
pasta.mkdir(exist_ok=True)

▶️ Peça a imagem da capa de um livro. Uma imagem é pedida como qualquer página:

In [ ]:
url_capa = "https://books.toscrape.com/media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"
capa = requests.get(url_capa, timeout=20)

▶️ Confira o tipo do conteúdo e o tamanho, em bytes, do que chegou em `content`:

In [ ]:
print(capa.headers["Content-Type"], "-", len(capa.content), "bytes")

O tipo é `image/jpeg`: os bytes são de uma imagem, e não de um texto.

▶️ Grave os bytes em um arquivo, com `write_bytes()`:

In [ ]:
(pasta / "capa.jpg").write_bytes(capa.content)

O número que aparece é a quantidade de bytes gravados.

▶️ Mostre a imagem salva no notebook:

In [ ]:
from IPython.display import Image

Image(filename="dados/capa.jpg")

### JSON: dados já organizados

Muitos sites oferecem dados em **JSON**, um formato de texto que representa listas e dicionários. O quotes.toscrape.com tem um endereço que devolve as citações nesse formato.

▶️ Peça a primeira página desse endereço e confira o tipo do conteúdo:

In [ ]:
api = requests.get("https://quotes.toscrape.com/api/quotes?page=1", timeout=20)
api.headers["Content-Type"]

O tipo é `application/json`.

▶️ O método `json()` converte o conteúdo em um dicionário do Python:

In [ ]:
dados = api.json()
type(dados)

▶️ Veja as chaves do dicionário:

In [ ]:
list(dados.keys())

A chave `quotes` guarda a lista de citações, e `has_next` diz se existe uma próxima página.

▶️ Pegue a primeira citação da lista e mostre o texto e o autor:

In [ ]:
primeira = dados["quotes"][0]
print(primeira["text"])
print("-", primeira["author"]["name"])

Os dados já vêm separados em campos, sem precisar procurar nada no HTML. No Módulo 07, você vai aprender a descobrir endereços como esse.

### Sua vez: lendo JSON

**Passo 1 de 4.** Peça a página 2 do mesmo endereço (`https://quotes.toscrape.com/api/quotes?page=2`) e guarde em `api2`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
api2 = requests.get("https://quotes.toscrape.com/api/quotes?page=2", timeout=20)
```
</details>

**Passo 2 de 4.** Converta a resposta em um dicionário chamado `dados2`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
dados2 = api2.json()
```
</details>

**Passo 3 de 4.** Mostre quantas citações vieram e se existe uma próxima página.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
print("citações:", len(dados2["quotes"]))
print("próxima página?", dados2["has_next"])
```
</details>

**Passo 4 de 4.** Mostre o nome do autor de cada citação.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
for citacao in dados2["quotes"]:
    print(citacao["author"]["name"])
```
</details>

## 1.3 Codificação: por que o £ virou Â£

No Módulo 02, o preço do primeiro livro apareceu como `Â£51.77`. O servidor envia **bytes**, e o `text` é o resultado de convertê-los em texto usando uma **codificação**. Quando o servidor não informa qual é (o `charset` do `Content-Type`), o requests assume ISO-8859-1, e a conversão sai errada:

<img src="imagens/codificacao.png" alt="Os bytes C2 A3 do símbolo da libra lidos como ISO-8859-1 viram Â£, e lidos como UTF-8 viram £" width="1100">

▶️ A codificação usada pelo requests fica em `encoding`. A variável `resposta` ainda guarda a página inicial do books.toscrape.com:

In [ ]:
resposta.encoding

▶️ O atributo `apparent_encoding` mostra a codificação que o requests **detecta** analisando os próprios bytes:

In [ ]:
resposta.apparent_encoding

As duas são diferentes: o requests usou ISO-8859-1, mas o conteúdo é UTF-8.

▶️ Veja o efeito no preço do primeiro livro. O método `find()` devolve a posição do trecho `price_color` no texto:

In [ ]:
posicao = resposta.text.find("price_color")
resposta.text[posicao:posicao + 25]

▶️ Corrija a codificação, informando a certa em `encoding`:

In [ ]:
resposta.encoding = "utf-8"

▶️ Mostre o preço de novo. A posição é procurada outra vez, porque o texto mudou:

In [ ]:
posicao = resposta.text.find("price_color")
resposta.text[posicao:posicao + 25]

Agora o preço aparece como `£51.77`. A grande maioria dos sites atuais usa UTF-8; na dúvida, use o valor de `apparent_encoding`.

> 💡 Os acentos do português sofrem o mesmo problema: `ação` vira `aÃ§Ã£o`. Sempre que aparecer um `Ã` ou um `Â` fora de lugar, desconfie da codificação.

### Sua vez: corrigindo a codificação

**Passo 1 de 3.** Peça a página do livro *A Light in the Attic* (`https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html`) e guarde em `livro`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
livro = requests.get("https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html", timeout=20)
```
</details>

**Passo 2 de 3.** Corrija a codificação de `livro`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
livro.encoding = "utf-8"
```
</details>

**Passo 3 de 3.** Mostre o trecho do preço, já com o símbolo correto.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
posicao = livro.text.find("price_color")
livro.text[posicao:posicao + 25]
```
</details>

> **Referências da Parte 1:** [Quickstart](https://requests.readthedocs.io/en/latest/user/quickstart/) da documentação do requests (seções *Make a Request*, *Response Content*, *Binary Response Content* e *JSON Response Content*, em inglês); [Content-Type](https://developer.mozilla.org/pt-BR/docs/Web/HTTP/Reference/Headers/Content-Type) e [JSON](https://developer.mozilla.org/pt-BR/docs/Glossary/JSON) (MDN); [Unicode](https://docs.python.org/pt-br/3/howto/unicode.html) e [pathlib](https://docs.python.org/pt-br/3/library/pathlib.html) (documentação do Python).

---

# Parte 2: Status e erros

Um scraper faz centenas de pedidos, e alguns vão dar errado. Há dois tipos de problema: o servidor responde com um código de erro, ou nenhuma resposta chega. Cada caso pede uma reação diferente:

<img src="imagens/fluxo-tratamento.png" alt="À esquerda, as exceções quando não chega resposta: ConnectionError, Timeout e RequestException; à direita, os códigos 200, 404, 403, 429 e 5xx e o que fazer em cada um" width="1120">

## 2.1 Conferindo o status

▶️ Peça uma página que não existe, a 51 do catálogo:

In [ ]:
pagina51 = requests.get("https://books.toscrape.com/catalogue/page-51.html", timeout=20)

▶️ Compare o `status_code` com `if` e `elif` para decidir o que fazer:

In [ ]:
if pagina51.status_code == 200:
    print("A página existe")
elif pagina51.status_code == 404:
    print("A página não existe")
else:
    print("Outro resultado:", pagina51.status_code)

Para testar várias páginas sem repetir esse código, ele pode virar uma **função**, que recebe a URL, faz o pedido e descreve o resultado.

▶️ Crie a função `verificar()`:

In [ ]:
def verificar(url):
    resposta = requests.get(url, timeout=20)
    if resposta.status_code == 200:
        print("200: a página existe     ", url)
    elif resposta.status_code == 404:
        print("404: a página não existe ", url)
    else:
        print(resposta.status_code, ": outro resultado ", url)

▶️ Use a função com as páginas 50 e 51:

In [ ]:
verificar("https://books.toscrape.com/catalogue/page-50.html")
verificar("https://books.toscrape.com/catalogue/page-51.html")

### Sua vez: 200 não garante conteúdo

**Passo 1 de 3.** Use `verificar()` com as páginas `https://quotes.toscrape.com/page/10/` e `https://quotes.toscrape.com/page/11/`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
verificar("https://quotes.toscrape.com/page/10/")
verificar("https://quotes.toscrape.com/page/11/")
```
</details>

**Passo 2 de 3.** As duas dão `200`. Peça a página 11 e guarde em `pagina11`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
pagina11 = requests.get("https://quotes.toscrape.com/page/11/", timeout=20)
```
</details>

**Passo 3 de 3.** Confira se o texto de `pagina11` contém a frase `No quotes found!`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
"No quotes found!" in pagina11.text
```
</details>

A página 11 não existe de verdade, mas o site responde `200` com uma mensagem. Por isso, além do status, um scraper precisa conferir se a página trouxe o conteúdo esperado.

## 2.2 raise_for_status: transformando o erro em exceção

O método `raise_for_status()` não faz nada quando o pedido deu certo, mas lança uma **exceção** (um erro que interrompe o programa) quando o status é `4xx` ou `5xx`.

▶️ Chame `raise_for_status()` na resposta da página 51. Vai aparecer um erro, **de propósito**:

In [ ]:
pagina51.raise_for_status()

A última linha do erro diz `HTTPError: 404 Client Error: Not Found for url: ...`. Para que o programa não pare, a exceção é capturada com `try` e `except`: o código em `try` é tentado e, se a exceção acontecer, o programa continua pelo `except`.

▶️ Execute:

In [ ]:
try:
    pagina51.raise_for_status()
    print("Deu certo")
except requests.exceptions.HTTPError as erro:
    print("Erro HTTP:", erro)

Agora o erro virou uma mensagem, e o programa seguiria normalmente para as próximas linhas.

## 2.3 Redirecionamentos

O requests segue os redirecionamentos (`3xx`) automaticamente.

▶️ Peça o endereço `/pages/simple` do scrapethissite, **sem** a barra final:

In [ ]:
simples = requests.get("https://www.scrapethissite.com/pages/simple", timeout=20)

▶️ O atributo `url` mostra o endereço **final**, depois do redirecionamento:

In [ ]:
print(simples.url, simples.status_code)

O requests chegou em `/pages/simple/`, com a barra, e recebeu `200`. O endereço final usa `http://` porque foi esse o endereço que o próprio servidor indicou.

▶️ O atributo `history` guarda as respostas intermediárias:

In [ ]:
simples.history

O primeiro endereço respondeu `308`, um redirecionamento permanente.

## 2.4 Erros de rede e tempo esgotado

Quando nenhuma resposta chega, não existe `status_code`: o requests lança uma exceção. As mais comuns são `ConnectionError` (sem internet, ou um site que não existe) e `Timeout` (o servidor demorou mais que o `timeout`).

▶️ Peça um site que não existe. O domínio `.invalid` é reservado e nunca existe, então vai aparecer um erro, **de propósito**:

In [ ]:
requests.get("https://site-que-nao-existe.invalid/", timeout=10)

O erro é longo, mas a última linha diz `ConnectionError`.

▶️ Capture a exceção com `try/except`:

In [ ]:
try:
    requests.get("https://site-que-nao-existe.invalid/", timeout=10)
except requests.exceptions.ConnectionError:
    print("Falha de conexão: o site não existe ou não respondeu")

▶️ Agora o tempo esgotado. Um `timeout` de 0,001 segundo é curto demais para qualquer servidor:

In [ ]:
try:
    requests.get("https://books.toscrape.com/", timeout=0.001)
except requests.exceptions.Timeout:
    print("Tempo esgotado")

## 2.5 Juntando tudo em uma função

A função `baixar()` reúne os tratamentos desta parte. Ela:

1. faz o pedido e chama `raise_for_status()`;
2. se o status for de erro (`HTTPError`), mostra a mensagem e devolve `None`, que significa "nada";
3. se nenhuma resposta chegar, mostra o tipo do erro e também devolve `None`. A exceção `RequestException` é a "mãe" de todas as exceções do requests, e por isso captura qualquer erro de rede;
4. se tudo der certo, corrige a codificação e devolve o texto da página.

▶️ Crie a função:

In [ ]:
def baixar(url):
    """Baixa uma página e devolve o texto, ou None se algo der errado."""
    try:
        resposta = requests.get(url, timeout=20)
        resposta.raise_for_status()
    except requests.exceptions.HTTPError as erro:
        print("Erro HTTP:", erro)
        return None
    except requests.exceptions.RequestException as erro:
        print("Erro de rede:", type(erro).__name__, "em", url)
        return None
    resposta.encoding = "utf-8"
    return resposta.text

▶️ Teste com uma página que existe:

In [ ]:
html = baixar("https://books.toscrape.com/catalogue/page-2.html")
len(html)

▶️ E com uma que não existe:

In [ ]:
html = baixar("https://books.toscrape.com/catalogue/page-60.html")
print(html)

### Sua vez: usando a função baixar()

**Passo 1 de 2.** Crie uma lista chamada `enderecos` com estes quatro endereços: a página 1 do catálogo (`https://books.toscrape.com/catalogue/page-1.html`), a página 60 do catálogo, um site `.invalid` e a página 3 do quotes (`https://quotes.toscrape.com/page/3/`).

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
enderecos = [
    "https://books.toscrape.com/catalogue/page-1.html",
    "https://books.toscrape.com/catalogue/page-60.html",
    "https://site-que-nao-existe.invalid/",
    "https://quotes.toscrape.com/page/3/",
]
```
</details>

**Passo 2 de 2.** Percorra a lista com um `for`, chamando `baixar()` para cada endereço, e conte quantos foram baixados com sucesso (os que não devolveram `None`).

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
sucessos = 0
for url in enderecos:
    if baixar(url) is not None:
        sucessos += 1
print("Baixados com sucesso:", sucessos)
```
</details>

Dois endereços dão certo; a página 60 dá `404`, e o domínio `.invalid` dá erro de rede.

> **Referências da Parte 2:** [Response Status Codes, Redirection and History, Timeouts e Errors and Exceptions](https://requests.readthedocs.io/en/latest/user/quickstart/) no Quickstart do requests, e a lista de [exceções](https://requests.readthedocs.io/en/latest/api/#exceptions) (em inglês); [Erros e exceções](https://docs.python.org/pt-br/3/tutorial/errors.html) (tutorial do Python); [Códigos de status de respostas HTTP](https://developer.mozilla.org/pt-BR/docs/Web/HTTP/Reference/Status) (MDN).

---

# Parte 3: Personalizando o pedido

## 3.1 Parâmetros na URL

Os parâmetros (a parte depois do `?`) podem ser escritos à mão na URL, mas é mais seguro passá-los em um dicionário, no argumento `params`. O requests monta a URL e converte espaços e acentos para o formato correto.

▶️ Monte o dicionário com os parâmetros da busca de times do scrapethissite:

In [ ]:
busca = {"q": "new york", "page_num": 2}

▶️ Faça o pedido passando o dicionário em `params`:

In [ ]:
times = requests.get("https://www.scrapethissite.com/pages/forms/", params=busca, timeout=20)

▶️ Veja a URL que o requests montou:

In [ ]:
times.url

O espaço virou `+`, e os dois parâmetros foram unidos com `&`.

▶️ Conte quantas linhas de times vieram nesta página do resultado:

In [ ]:
times.text.count('<tr class="team">')

### Sua vez: buscando com params

**Passo 1 de 3.** Crie um dicionário `busca2` para buscar os times com `rangers` no nome.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
busca2 = {"q": "rangers"}
```
</details>

**Passo 2 de 3.** Faça o pedido com `params=busca2` e guarde em `rangers`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
rangers = requests.get("https://www.scrapethissite.com/pages/forms/", params=busca2, timeout=20)
```
</details>

**Passo 3 de 3.** Mostre a URL montada e quantas linhas de times vieram.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
print(rangers.url)
print(rangers.text.count('<tr class="team">'))
```
</details>

## 3.2 Cabeçalhos e User-Agent

O argumento `headers` recebe um dicionário com os cabeçalhos a enviar. O mais importante para o scraping é o `User-Agent`, que identifica o programa que faz o pedido. Alguns sites recusam certos programas.

▶️ Faça um pedido ao scrapethissite sem cabeçalhos e veja o `User-Agent` que o requests enviou:

In [ ]:
url_simples = "https://www.scrapethissite.com/pages/simple/"

padrao = requests.get(url_simples, timeout=20)
print(padrao.status_code, padrao.request.headers["User-Agent"])

▶️ Agora o mesmo pedido, dizendo que o programa é o `Python-urllib`, outra biblioteca do Python:

In [ ]:
urllib = requests.get(url_simples, headers={"User-Agent": "Python-urllib/3.14"}, timeout=20)
urllib.status_code

O site recusou: `403 Forbidden`. É o mesmo motivo pelo qual o leitor de robots.txt do Python falhou no Módulo 02.

Em projetos reais, a boa prática é usar um `User-Agent` que diga quem você é e como entrar em contato. Se um site recusar o seu scraper, isso é um sinal para rever as regras dele, e não para disfarçar o programa.

▶️ Faça o pedido com um `User-Agent` que identifica o scraper:

In [ ]:
identificado = requests.get(url_simples, headers={"User-Agent": "curso-web-scraping (aluno@exemplo.com)"}, timeout=20)
identificado.status_code

### Sua vez: enviando cabeçalhos

**Passo 1 de 3.** Crie um dicionário `cabecalhos` com dois cabeçalhos: um `User-Agent` com o seu nome e `Accept-Language` com o valor `pt-BR`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
cabecalhos = {"User-Agent": "curso-web-scraping (seu-nome)", "Accept-Language": "pt-BR"}
```
</details>

**Passo 2 de 3.** Faça um pedido para `https://books.toscrape.com/` com esses cabeçalhos e guarde em `com_cabecalhos`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
com_cabecalhos = requests.get("https://books.toscrape.com/", headers=cabecalhos, timeout=20)
```
</details>

**Passo 3 de 3.** Mostre os cabeçalhos que foram enviados, que ficam em `com_cabecalhos.request.headers`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
for nome, valor in com_cabecalhos.request.headers.items():
    print(nome, ":", valor)
```
</details>

## 3.3 Cookies

O HTTP não guarda memória entre um pedido e outro. Para "lembrar" de um visitante (um login, um carrinho de compras), o servidor envia um **cookie** no cabeçalho `Set-Cookie`, e o cliente o devolve nos pedidos seguintes, no cabeçalho `Cookie`:

<img src="imagens/cookies-sessao.png" alt="Diagrama do login no quotes.toscrape.com: o servidor envia o cookie session, a Session o guarda e o devolve no POST e nos pedidos seguintes" width="1100">

▶️ Peça a página de login do quotes.toscrape.com:

In [ ]:
URL_LOGIN = "https://quotes.toscrape.com/login"

formulario = requests.get(URL_LOGIN, timeout=20)

▶️ Veja o cabeçalho `Set-Cookie` que o servidor enviou (só o começo, porque ele é longo):

In [ ]:
formulario.headers["Set-Cookie"][:60]

▶️ Os cookies recebidos também ficam organizados em `cookies`. Veja o nome deles:

In [ ]:
list(formulario.cookies.keys())

O servidor enviou um cookie chamado `session`.

## 3.4 Fazendo login, primeiro sem sessão

O formulário de login exige três campos: usuário, senha (qualquer valor serve, é um site de treino) e um código escondido, o `csrf_token`. Esse código só é aceito **junto com o cookie `session`** que veio com o formulário.

▶️ A função abaixo lê o `csrf_token` do formulário com o `parsel`, como na prévia da seção 1.1. Crie a função e use-a no formulário que você já baixou:

In [ ]:
def pegar_token(html):
    """Lê o valor do campo escondido csrf_token do formulário."""
    return Selector(text=html).css("input[name=csrf_token]::attr(value)").get()

token = pegar_token(formulario.text)
token[:20]

▶️ Monte um dicionário com os três campos do formulário:

In [ ]:
login = {"csrf_token": token, "username": "aluno", "password": "123"}

▶️ Envie o login com `requests.post()`. Os campos do formulário vão no argumento `data`:

In [ ]:
tentativa = requests.post(URL_LOGIN, data=login, timeout=20)

▶️ Confira se o login funcionou (a página de quem está logado tem o link **Logout**) e leia a mensagem de erro do site:

In [ ]:
print("Logado?", "Logout" in tentativa.text)
print(Selector(text=tentativa.text).css("p.error::text").get())

O login falhou: *invalid CRSF token*. O token foi enviado, mas o cookie que o acompanhava ficou para trás, no primeiro pedido: cada `requests.get()` ou `requests.post()` é independente.

## 3.5 Fazendo login com uma sessão

Uma **sessão** (`requests.Session`) guarda os cookies recebidos e os devolve sozinha nos pedidos seguintes. Os cabeçalhos definidos nela também valem para todos os pedidos.

▶️ Crie a sessão e defina o `User-Agent` dela:

In [ ]:
sessao = requests.Session()
sessao.headers["User-Agent"] = "curso-web-scraping (aluno@exemplo.com)"

▶️ Peça o formulário **pela sessão** (`sessao.get`, e não `requests.get`):

In [ ]:
formulario = sessao.get(URL_LOGIN, timeout=20)

▶️ A sessão já guardou o cookie recebido:

In [ ]:
list(sessao.cookies.keys())

▶️ Leia o token do novo formulário e monte o login de novo:

In [ ]:
login = {"csrf_token": pegar_token(formulario.text), "username": "aluno", "password": "123"}

▶️ Envie o login pela sessão. Ela inclui o cookie automaticamente:

In [ ]:
logado = sessao.post(URL_LOGIN, data=login, timeout=20)

▶️ Confira o resultado:

In [ ]:
print("Logado?", "Logout" in logado.text)
print("Caminho:", logado.history, "->", logado.url)

O login funcionou: o servidor respondeu com um redirecionamento (`302`) para a página inicial, que agora mostra o link **Logout**. A sessão continua aberta e logada, e será usada no exercício a seguir.

### Sua vez: o cookie é o que mantém o login

**Passo 1 de 4.** Peça a página inicial (`https://quotes.toscrape.com/`) **pela sessão** e conte quantas vezes aparece a palavra `Goodreads`, um link que só aparece para quem está logado.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
inicial = sessao.get("https://quotes.toscrape.com/", timeout=20)
inicial.text.count("Goodreads")
```
</details>

**Passo 2 de 4.** Agora copie o valor do cookie de login para uma variável: ele fica em `sessao.cookies["session"]`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
valor = sessao.cookies["session"]
```
</details>

**Passo 3 de 4.** Faça um pedido **comum** (`requests.get`, sem sessão) para a página inicial, passando o cookie à mão no argumento `cookies`, como um dicionário: `cookies={"session": valor}`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
com_cookie = requests.get("https://quotes.toscrape.com/", cookies={"session": valor}, timeout=20)
```
</details>

**Passo 4 de 4.** Confira se esse pedido comum também está logado (se o texto contém `Logout`).

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
"Logout" in com_cookie.text
```
</details>

O pedido comum, sem sessão, também chegou logado: é o **cookie** que identifica o login. A sessão apenas guarda e devolve os cookies por você.

▶️ No final, feche a sessão. Isso encerra as conexões abertas com o servidor:

In [ ]:
sessao.close()

> **Referências da Parte 3:** [Passing Parameters In URLs, Custom Headers e Cookies](https://requests.readthedocs.io/en/latest/user/quickstart/) no Quickstart, e [Session Objects](https://requests.readthedocs.io/en/latest/user/advanced/#session-objects) no guia avançado do requests (em inglês); [User-Agent](https://developer.mozilla.org/pt-BR/docs/Web/HTTP/Reference/Headers/User-Agent), [Cookies HTTP](https://developer.mozilla.org/pt-BR/docs/Web/HTTP/Guides/Cookies) e [Set-Cookie](https://developer.mozilla.org/pt-BR/docs/Web/HTTP/Reference/Headers/Set-Cookie) (MDN); [CSRF](https://developer.mozilla.org/en-US/docs/Glossary/CSRF) (MDN, em inglês), sobre o motivo do `csrf_token`.

---

# Parte 4: Missão: baixar o catálogo inteiro

## 4.1 Pausas entre os pedidos

Um scraper consegue fazer dezenas de pedidos por segundo, o que pode sobrecarregar um site. A função `time.sleep()` pausa o programa pelo número de segundos indicado, e é a forma mais simples de manter um ritmo responsável.

▶️ Peça as três primeiras páginas do catálogo, com uma pausa de 1 segundo entre elas. O endereço de cada página é montado com uma f-string, trocando apenas o número, e `range(1, 4)` gera os números 1, 2 e 3:

In [ ]:
import time

for numero in range(1, 4):
    resposta = requests.get(f"https://books.toscrape.com/catalogue/page-{numero}.html", timeout=20)
    print(f"página {numero}: {resposta.status_code}")
    time.sleep(1)

## 4.2 A missão

Agora, o catálogo inteiro. O programa não sabe quantas páginas existem: ele pede página após página até receber um `404`, que indica o fim. Cada página é salva em um arquivo na pasta `dados/catalogo`, para ser usada no Módulo 04.

<img src="imagens/coleta-catalogo.png" alt="O laço da coleta: montar o endereço, pedir com a sessão, decidir pelo status, salvar o HTML e esperar; a página 51 dá 404 e encerra o laço" width="1100">

A missão é montada em partes: primeiro as peças, testadas uma a uma, e depois o laço que as junta.

▶️ Crie a pasta onde as páginas serão salvas:

In [ ]:
pasta_catalogo = Path("dados/catalogo")
pasta_catalogo.mkdir(parents=True, exist_ok=True)

▶️ Crie uma sessão para a missão, com um `User-Agent` que identifica o scraper:

In [ ]:
sessao = requests.Session()
sessao.headers["User-Agent"] = "curso-web-scraping (aluno@exemplo.com)"

▶️ Crie a função que pede uma página do catálogo pelo número e já corrige a codificação:

In [ ]:
def pedir_pagina(numero):
    resposta = sessao.get(f"https://books.toscrape.com/catalogue/page-{numero}.html", timeout=20)
    resposta.encoding = "utf-8"
    return resposta

▶️ Teste a função com a página 1 e com a 51:

In [ ]:
print(pedir_pagina(1).status_code, pedir_pagina(51).status_code)

▶️ Crie a função que salva uma página em um arquivo. O `{numero:02d}` escreve o número com dois dígitos (`01`, `02`...), para que os arquivos fiquem em ordem:

In [ ]:
def salvar(resposta, numero):
    arquivo = pasta_catalogo / f"pagina-{numero:02d}.html"
    arquivo.write_text(resposta.text, encoding="utf-8")
    return arquivo

▶️ Teste a função salvando a página 1:

In [ ]:
salvar(pedir_pagina(1), 1)

▶️ Agora o laço que junta as peças. O `while True` repete até encontrar o `break`, que acontece no `404`; qualquer outro erro é interrompido pelo `raise_for_status()`. A coleta leva cerca de 40 segundos:

In [ ]:
numero = 1
while True:
    resposta = pedir_pagina(numero)
    if resposta.status_code == 404:
        print(f"Página {numero}: 404. Fim do catálogo.")
        break
    resposta.raise_for_status()
    salvar(resposta, numero)
    if numero % 10 == 0:
        print(f"{numero} páginas salvas...")
    numero += 1
    time.sleep(0.5)

▶️ Feche a sessão da missão:

In [ ]:
sessao.close()

▶️ Confira quantos arquivos foram salvos:

In [ ]:
arquivos = sorted(pasta_catalogo.glob("*.html"))
print(len(arquivos), "arquivos, de", arquivos[0].name, "a", arquivos[-1].name)

▶️ Conte os livros: cada um fica dentro de um bloco com `class="product_pod"`.

In [ ]:
sum(arquivo.read_text(encoding="utf-8").count('class="product_pod"') for arquivo in arquivos)

São 50 arquivos e 1000 livros. O catálogo inteiro está no seu computador, e o Módulo 04 vai extrair dele o título, o preço e a nota de cada livro, sem precisar baixar tudo de novo.

### Sua vez: a missão no quotes.toscrape.com

**Passo 1 de 5.** Crie a pasta `dados/citacoes`, guardada na variável `pasta_citacoes`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
pasta_citacoes = Path("dados/citacoes")
pasta_citacoes.mkdir(parents=True, exist_ok=True)
```
</details>

**Passo 2 de 5.** Crie uma função `pedir_citacoes(numero)` que pede a página `https://quotes.toscrape.com/page/{numero}/` e devolve a resposta.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
def pedir_citacoes(numero):
    resposta = requests.get(f"https://quotes.toscrape.com/page/{numero}/", timeout=20)
    resposta.encoding = "utf-8"
    return resposta
```
</details>

**Passo 3 de 5.** Teste a função com a página 11 e confira que ela contém `No quotes found!`: esse site **não** responde `404` no fim, e esse texto será o critério de parada.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
"No quotes found!" in pedir_citacoes(11).text
```
</details>

**Passo 4 de 5.** Escreva o laço: peça página após página, pare quando aparecer `No quotes found!` e salve cada página em `pasta_citacoes / f"pagina-{numero:02d}.html"`, com uma pausa de 0,5 segundo.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
numero = 1
while True:
    resposta = pedir_citacoes(numero)
    if "No quotes found!" in resposta.text:
        print(f"Página {numero} vazia. Fim.")
        break
    (pasta_citacoes / f"pagina-{numero:02d}.html").write_text(resposta.text, encoding="utf-8")
    numero += 1
    time.sleep(0.5)
```
</details>

**Passo 5 de 5.** Confira quantas páginas foram salvas.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
len(list(pasta_citacoes.glob("*.html")))
```
</details>

São 10 páginas com citações. O critério de parada mudou de "status 404" para "página sem conteúdo", a lição do exercício da seção 2.1.

> **Referências da Parte 4:** [time.sleep](https://docs.python.org/pt-br/3/library/time.html#time.sleep) e [pathlib](https://docs.python.org/pt-br/3/library/pathlib.html) (documentação do Python); [Session Objects](https://requests.readthedocs.io/en/latest/user/advanced/#session-objects) (documentação do requests, em inglês).

---

## Problemas comuns e como resolver

| Sintoma | Causa provável | Solução |
|---|---|---|
| `NameError: name 'requests' is not defined` (ou outro nome) | Uma célula anterior não foi executada | Execute as células em ordem, de cima para baixo. Na dúvida: **Restart Kernel and Run All** |
| Aparecem `Â£`, `Ã§`, `Ã£` no texto | Codificação errada | `resposta.encoding = "utf-8"` antes de ler o `text` (seção 1.3) |
| `JSONDecodeError` ao usar `json()` | A resposta não é JSON (por exemplo, uma página de erro em HTML) | Confira o `status_code` e o `Content-Type` antes |
| `ConnectionError` | Sem internet, endereço digitado errado ou site fora do ar | Confira a conexão e o endereço; abra-o no navegador |
| `Timeout` ou `ReadTimeout` | O servidor demorou mais que o `timeout` | Tente de novo; se for frequente, aumente o `timeout` |
| `403 Forbidden` | O site recusou o pedido, muitas vezes pelo `User-Agent` | Use um `User-Agent` que identifique o scraper e confira as regras do site |
| O login não funciona (`invalid CRSF token`) | O cookie não foi devolvido ao servidor | Faça todos os pedidos pela mesma sessão (seção 3.5) |
| A célula fica em `[*]` por muito tempo | A missão em andamento, ou um pedido sem `timeout` | A missão leva cerca de 40 s. Se passar disso, interrompa com ■ e confira se todos os pedidos têm `timeout` |

---

# Retomada do Módulo 03

<img src="imagens/retomada.png" alt="Mapa de revisão do módulo: a resposta, status, erros de rede, parâmetros e cabeçalhos, cookies e sessão, coleta responsável" width="1100">

### O que você aprendeu

- `resposta = requests.get(url, timeout=...)` guarda na variável **tudo o que voltou do servidor**, organizado em atributos: `status_code`, `headers`, `text`, `content`, `json()`, `url`, `history`, `cookies` e até o pedido original, em `request`.
- O `text` é o HTML da página. No Módulo 04, ele será entregue ao parsel para que os dados sejam extraídos pela estrutura do HTML.
- Quando o servidor não informa a codificação, defina `resposta.encoding = "utf-8"` antes de ler o texto.
- Confira o **status** com `status_code` ou com `raise_for_status()`. Um `200` não garante conteúdo: confira também se a página trouxe o que você esperava.
- Erros de rede viram **exceções** (`ConnectionError`, `Timeout`), tratadas com `try/except`. Todo pedido deve ter `timeout`.
- `params` monta a URL com segurança; `headers` envia cabeçalhos, como um `User-Agent` que identifica o scraper.
- O **cookie** é o que mantém um login. Uma `requests.Session()` guarda os cookies e os cabeçalhos e os envia em todos os pedidos.
- Uma coleta responsável faz **pausas** entre os pedidos e sabe quando parar.

### Teste seus conhecimentos

Clique em cada pergunta para ver a resposta.

<details>
<summary><b>1.</b> Depois de <code>r = requests.get(url, timeout=20)</code>, onde ficam o código de status, o HTML da página e o pedido que foi enviado?</summary>

Todos dentro de `r`, em atributos: `r.status_code`, `r.text` e `r.request`.
</details>

<details>
<summary><b>2.</b> Qual atributo da resposta você usa para salvar uma imagem em disco, e por quê?</summary>

`resposta.content`, que devolve os **bytes** do arquivo. O `text` tentaria converter os bytes em texto, o que não faz sentido para uma imagem.
</details>

<details>
<summary><b>3.</b> O texto baixado mostra <code>InformaÃ§Ãµes</code>. Qual é o problema e como resolver?</summary>

A codificação foi adivinhada errado. Defina `resposta.encoding = "utf-8"` antes de ler o `resposta.text`.
</details>

<details>
<summary><b>4.</b> Qual é a diferença entre receber um <code>404</code> e receber um <code>ConnectionError</code>?</summary>

No `404`, o servidor **respondeu**, dizendo que a página não existe; a resposta tem um `status_code`. No `ConnectionError`, **nenhuma resposta chegou**, e o requests lança uma exceção.
</details>

<details>
<summary><b>5.</b> Por que o login do quotes.toscrape.com falha sem uma sessão?</summary>

Porque o `csrf_token` só é aceito junto com o cookie `session` enviado com o formulário. Sem sessão, o cookie não é devolvido no `POST`.
</details>

### Glossário

| Termo | Significado |
|---|---|
| **Response** | O objeto que o requests devolve, com tudo o que o servidor enviou |
| **Atributo** | Uma informação guardada dentro de um objeto, acessada com ponto: `resposta.text` |
| **Bytes** | O conteúdo bruto da resposta, antes de virar texto |
| **Codificação** (*encoding*) | A regra que converte bytes em texto, como UTF-8 ou ISO-8859-1 |
| **JSON** | Formato de texto que representa listas e dicionários, usado por APIs |
| **Exceção** | Um erro que interrompe o programa, a menos que seja tratado com `try/except` |
| **Timeout** | O tempo máximo de espera por uma resposta |
| **Parâmetros** (`params`) | Os pares `nome=valor` adicionados à URL |
| **User-Agent** | O cabeçalho que identifica o programa que faz o pedido |
| **Cookie** | Um dado que o servidor envia e o cliente devolve nos pedidos seguintes |
| **Sessão** (`Session`) | Um objeto do requests que guarda cookies e cabeçalhos entre pedidos |
| **CSRF token** | Um código de segurança de formulários, que só vale junto com o cookie da mesma visita |

---

## Próximo módulo

O **Módulo 04: Seletores XPath e CSS** apresenta o `parsel`, visto de relance nas seções 1.1 e 3.4. Com ele, o `text` deixa de ser uma longa sequência de caracteres e passa a ser lido pela estrutura do HTML: você vai extrair o título, o preço e a nota de cada um dos 1000 livros salvos em `dados/catalogo`.

## Referências

Materiais consultados na elaboração deste módulo, agrupados pela parte em que foram usados.

**Documentação do requests** (em inglês)

- [Quickstart](https://requests.readthedocs.io/en/latest/user/quickstart/): pedidos, conteúdo da resposta, JSON, parâmetros, cabeçalhos, status, redirecionamentos, cookies, timeouts e exceções (Partes 1 a 3).
- [Advanced Usage: Session Objects](https://requests.readthedocs.io/en/latest/user/advanced/#session-objects) (Partes 3 e 4).
- [Developer Interface](https://requests.readthedocs.io/en/latest/api/), com a lista de exceções (Parte 2).

**HTTP e formatos** (MDN Web Docs)

- [Content-Type](https://developer.mozilla.org/pt-BR/docs/Web/HTTP/Reference/Headers/Content-Type) e [JSON](https://developer.mozilla.org/pt-BR/docs/Glossary/JSON) (Parte 1).
- [Códigos de status de respostas HTTP](https://developer.mozilla.org/pt-BR/docs/Web/HTTP/Reference/Status) (Parte 2).
- [User-Agent](https://developer.mozilla.org/pt-BR/docs/Web/HTTP/Reference/Headers/User-Agent), [Cookies HTTP](https://developer.mozilla.org/pt-BR/docs/Web/HTTP/Guides/Cookies), [Set-Cookie](https://developer.mozilla.org/pt-BR/docs/Web/HTTP/Reference/Headers/Set-Cookie) e [CSRF](https://developer.mozilla.org/en-US/docs/Glossary/CSRF), este em inglês (Parte 3).

**Documentação do Python**

- [Unicode](https://docs.python.org/pt-br/3/howto/unicode.html) (Parte 1), [Erros e exceções](https://docs.python.org/pt-br/3/tutorial/errors.html) (Parte 2), [time](https://docs.python.org/pt-br/3/library/time.html) e [pathlib](https://docs.python.org/pt-br/3/library/pathlib.html) (Partes 1 e 4).

**Leitura complementar**

- SEVERANCE, Charles. *Python for Everybody*, [capítulo 12: Networked programs](https://www.py4e.com/html3/12-network). Em inglês. Do mesmo autor do curso *Django for Everybody*, citado no Módulo 02, mostra os mesmos conceitos com a biblioteca padrão do Python.

**Sites de treino usados nos exemplos:** [books.toscrape.com](https://books.toscrape.com), [quotes.toscrape.com](https://quotes.toscrape.com) e [scrapethissite.com](https://www.scrapethissite.com/pages/).